# Multiscale ridge filter (eigenvalues of the Hessian)

**Why:** thin, elongated structures (insect/arachnid legs and antennae, spider webs, leaf veins, fur/feather strands) are
*ridges*: intensity changes sharply across the line but barely along it. At a pixel, the Hessian's two eigenvalues
capture this: a line has one large |eigenvalue| (across) and one near zero (along), a blob has two large ones of the same
sign, and flat regions have two small ones. We compute this at several scales (sigma) so both thin legs and thicker veins respond.

**Per image we extract (35 features):**
- for each sigma in {1,2,3,4} and each polarity (dark ridge on light / bright ridge on dark): mean, std, 95th percentile, density
- orientation coherence (are ridges aligned? fur/feathers vs. random webs)
- radial alignment (do ridges point toward/away from the image center? spider legs, radial fungi)
- fine/coarse ridge-energy ratio

Saved to `features/ridge_features.npz` using the same row order, `y`, `idx_dev`, `idx_test` as `lbp_color_features.npz`.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-image scikit-learn pillow joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from joblib import Parallel, delayed
from skimage.feature import hessian_matrix, hessian_matrix_eigvals
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
SEED = 42
TARGET = 256                 # longest side after resize (keeps aspect ratio, so sigma means the same thing on every image)
SIGMAS = [1, 2, 3, 4]        # ridge scales in pixels: small = thin legs/antennae/fur, large = veins/thicker limbs
BETA = 0.5                   # blob suppression strength (smaller = stricter about "line-like")
LABEL_COLUMN = "supercategory"
META_PATH = Path("./sample_images/metadata.csv")
FEATURE_DIR = Path("./features")
FEATURE_DIR.mkdir(exist_ok=True)

IMG_DIR = Path("./sample_images/images")
if not IMG_DIR.exists():
    IMG_DIR = Path("../sample/images")

metadata = pd.read_csv(META_PATH, index_col=0)[["image_id", "file_name", "common_name", LABEL_COLUMN]]
y = metadata[LABEL_COLUMN].to_numpy()
categories = sorted(set(y))
print(len(metadata), "images,", len(categories), "classes, reading from", IMG_DIR.resolve())

## Ridge maps from Hessian eigenvalues

In [ ]:
def load_gray(path, target=TARGET):
    img = Image.open(path).convert("RGB")
    s = target / max(img.size)
    img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.LANCZOS)
    return np.asarray(img.convert("L"), dtype=np.float64) / 255.0


def ridge_maps(gray, sigmas=SIGMAS, beta=BETA):
    """Per sigma: 'dark' (dark line on light bg), 'bright' (light line on dark bg), 'theta_*' (orientation across the line).
    skimage returns eigenvalues sorted l1 >= l2.
      dark ridge  : l1 >> 0 and |l2| ~ 0   -> response |l1| * exp(-Rb^2 / 2beta^2),  Rb = |l2|/|l1|
      bright ridge: l2 << 0 and |l1| ~ 0   -> response |l2| * exp(-Rb^2 / 2beta^2),  Rb = |l1|/|l2|
    Rb is the Frangi blobness term: blobs have Rb ~ 1 and get suppressed. Responses are scaled by sigma^2 (scale normalisation)."""
    out, eps = [], 1e-12
    for s in sigmas:
        Hrr, Hrc, Hcc = hessian_matrix(gray, sigma=s, order="rc", use_gaussian_derivatives=False)
        l1, l2 = hessian_matrix_eigvals([Hrr, Hrc, Hcc])
        Rb_d = np.abs(l2) / (np.abs(l1) + eps)
        Rb_b = np.abs(l1) / (np.abs(l2) + eps)
        dark = np.where(l1 > 0, np.abs(l1) * np.exp(-Rb_d**2 / (2 * beta**2)), 0) * s**2
        bright = np.where(l2 < 0, np.abs(l2) * np.exp(-Rb_b**2 / (2 * beta**2)), 0) * s**2
        # angle (from the row axis toward the column axis) of the eigenvector of the LARGER eigenvalue
        theta = 0.5 * np.arctan2(2 * Hrc, Hrr - Hcc)
        # direction ACROSS the line: dark ridge -> l1's eigenvector; bright ridge -> l2's eigenvector (perpendicular to l1's)
        out.append(dict(sigma=s, dark=dark, bright=bright, theta_dark=theta, theta_bright=theta + np.pi / 2))
    return out


def max_over_scales(maps, key):
    return np.max([m[key] for m in maps], axis=0)

In [ ]:
def ridge_features(gray, sigmas=SIGMAS):
    maps = ridge_maps(gray, sigmas)
    f, names = [], []
    for m in maps:
        for pol in ("dark", "bright"):
            r = m[pol]
            thr = 0.02 * r.max() + 1e-12                       # relative threshold -> "ridge density"
            f += [r.mean(), r.std(), np.percentile(r, 95), (r > thr).mean()]
            names += [f"{pol}_s{m['sigma']}_{k}" for k in ("mean", "std", "p95", "density")]

    # orientation features from the strongest scale at each pixel
    resp = np.stack([m["dark"] for m in maps] + [m["bright"] for m in maps])
    thetas = np.stack([m["theta_dark"] for m in maps] + [m["theta_bright"] for m in maps])
    best = resp.argmax(0)
    w = resp.max(0)
    theta = np.take_along_axis(thetas, best[None], 0)[0]      # across-line angle of the strongest response
    wsum = w.sum() + 1e-12
    coherence = np.abs((w * np.exp(2j * theta)).sum()) / wsum        # 1 = all ridges parallel, 0 = isotropic
    h, wd = gray.shape
    rr, cc = np.mgrid[0:h, 0:wd]
    phi = np.arctan2(cc - wd / 2, rr - h / 2)                         # polar angle about the center, same convention as theta (from row axis toward column axis)
    radial = -(w * np.cos(2 * (theta - phi))).sum() / wsum            # >0 radial lines, <0 concentric/tangential
    fine = np.mean([m["dark"].mean() + m["bright"].mean() for m in maps[:2]])
    coarse = np.mean([m["dark"].mean() + m["bright"].mean() for m in maps[2:]])
    f += [coherence, radial, fine / (coarse + 1e-12)]
    names += ["orient_coherence", "radial_alignment", "fine_coarse_ratio"]
    return np.array(f, dtype=np.float32), names


def extract_one(rel_path):
    return ridge_features(load_gray(IMG_DIR / rel_path))[0]

_, feat_names = ridge_features(np.random.default_rng(0).random((64, 64)))
print(len(feat_names), "features")

## Illustration: ridge maps on example images (one per class)

In [ ]:
examples = metadata.groupby(LABEL_COLUMN).sample(n=1, random_state=1)
fig, axs = plt.subplots(len(examples), 3 + len(SIGMAS), figsize=(2.3 * (3 + len(SIGMAS)), 2.3 * len(examples)))
for ax_row, (_, row) in zip(axs, examples.iterrows()):
    g = load_gray(IMG_DIR / row["file_name"])
    maps = ridge_maps(g)
    panels = [(g, "gray", row[LABEL_COLUMN]),
              (max_over_scales(maps, "dark"), "magma", "dark ridges (max over scales)"),
              (max_over_scales(maps, "bright"), "magma", "bright ridges (max over scales)")]
    panels += [(m["dark"], "magma", f"dark, sigma={m['sigma']}") for m in maps]
    for ax, (im, cmap, t) in zip(ax_row, panels):
        ax.imshow(im, cmap=cmap, vmax=np.percentile(im, 99.5) if cmap == "magma" else None)
        ax.set_title(t, fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## Extract features for all images

In [ ]:
rows = Parallel(n_jobs=-1)(delayed(extract_one)(p) for p in metadata["file_name"])
X = np.vstack(rows)
print(X.shape, "NaNs:", int(np.isnan(X).sum()))

## Variation across classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, pol in zip(axes, ("dark", "bright")):
    width = 0.8 / len(SIGMAS)
    for i, s in enumerate(SIGMAS):
        idx = feat_names.index(f"{pol}_s{s}_mean")
        ax.bar(np.arange(len(categories)) + i * width, [X[y == c, idx].mean() for c in categories], width, label=f"sigma={s}")
    ax.set_xticks(np.arange(len(categories)) + 0.4 - width / 2)
    ax.set_xticklabels(categories, rotation=45, ha="right")
    ax.set_title(f"Mean {pol}-ridge response by class and scale"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, key in zip(axes, ["orient_coherence", "radial_alignment", "fine_coarse_ratio", "dark_s1_density"]):
    idx = feat_names.index(key)
    ax.boxplot([X[y == c, idx] for c in categories], showfliers=False)
    ax.set_xticks(range(1, len(categories) + 1)); ax.set_xticklabels(categories, rotation=60, ha="right")
    ax.set_title(key)
plt.tight_layout(); plt.show()

## PCA and t-SNE

In [ ]:
Z = StandardScaler().fit_transform(X)
pca = PCA().fit(Z)
plt.figure(figsize=(5, 3.5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("# components"); plt.ylabel("cumulative variance"); plt.title("Ridge features: PCA variance"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

P = PCA(n_components=2).fit_transform(Z)
T = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca").fit_transform(Z)
cmap = plt.get_cmap("tab20")
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, E, t in zip(axes, (P, T), ("PCA", "t-SNE")):
    for i, c in enumerate(categories):
        m = y == c
        ax.scatter(E[m, 0], E[m, 1], s=10, color=cmap(i), label=c, alpha=.7)
    ax.set_title(f"{t} of multiscale ridge features")
axes[1].legend(fontsize=7, markerscale=2, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

## Quick sanity check: does it carry class information?
Same protocol as `lbp_features.ipynb` (5-fold CV on the dev split, logistic regression), alone and combined with color + LBP.

In [ ]:
prev = np.load(FEATURE_DIR / "lbp_color_features.npz", allow_pickle=True)
assert (prev["image_id"] == metadata["image_id"].to_numpy()).all(), "row order differs from lbp_color_features.npz"
idx_dev, idx_test = prev["idx_dev"], prev["idx_test"]

sets = {"ridge": X, "color": prev["color"], "color + lbp_blur": np.hstack([prev["color"], prev["lbp_blur"]]),
        "ridge + color": np.hstack([X, prev["color"]]),
        "ridge + color + lbp_blur": np.hstack([X, prev["color"], prev["lbp_blur"]])}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
make_clf = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
out = []
for name, F in sets.items():
    r = cross_validate(make_clf(), F[idx_dev], y[idx_dev], cv=cv, scoring=["accuracy", "f1_macro"], n_jobs=-1)
    out.append({"features": name, "dims": F.shape[1], "acc": r["test_accuracy"].mean(),
                "acc std": r["test_accuracy"].std(), "macro-F1": r["test_f1_macro"].mean()})
results = pd.DataFrame(out).set_index("features")
print(f"chance = {1 / len(categories):.3f}")
results.round(3)

In [ ]:
pred = cross_val_predict(make_clf(), X[idx_dev], y[idx_dev], cv=cv, n_jobs=-1)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y[idx_dev], pred, labels=categories, normalize="true"),
                       display_labels=categories).plot(ax=ax, xticks_rotation=45, colorbar=False, values_format=".2f")
ax.set_title("Ridge features only (logistic regression, 5-fold CV on dev)")
plt.tight_layout(); plt.show()

In [ ]:
np.savez(FEATURE_DIR / "ridge_features.npz", y=y, image_id=metadata["image_id"].to_numpy(),
         idx_dev=idx_dev, idx_test=idx_test, ridge=X, feature_names=np.array(feat_names))
print("Saved", FEATURE_DIR / "ridge_features.npz")